# RQ1 · Institutions and GDP per capita
### Do improvements in governance quality (rule of law, corruption control, government effectiveness) predict higher GDP per capita, after absorbing country and year fixed effects?

**Design.**
- Unit of analysis: country × year.
- Sample: all countries (excluding regional/income aggregates), 1996-2023
  (Worldwide Governance Indicators start in 1996).
- Outcome: `log(GDP per capita, constant 2015 US$)`.
- Regressors of interest: `RL.EST` (rule of law), `CC.EST` (control of
  corruption), `GE.EST` (government effectiveness).
- Controls: `log(population)`, trade share, investment share.
- Fixed effects: country + year (two-way).
- SEs: clustered at country.

**Why this design?** Governance and GDP are jointly determined  -  richer
countries can afford better institutions, better institutions raise
productivity. Two-way FE cannot fix reverse causality but it strips
level differences (rich-country institutions vs poor) and common shocks
(financial crises, pandemics). Effects should therefore be interpreted
as *within-country* changes: when a country's rule-of-law score
improves, does its per-capita income rise?

**Compute footprint.** ~200 countries × 28 years ≈ 5,000 rows. Runs
locally in seconds. OSG version bootstraps SEs and adds a Hausman-style
test for RE vs FE, which is memory-heavy.


In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.wb_client import fetch_indicators, fetch_country_metadata, RQ1_INDICATORS
from src.regression import fixed_effects_panel


## 1. Pull the panel

In [ ]:
panel = fetch_indicators(
    indicators=list(RQ1_INDICATORS.keys()),
    countries="all",
    start=1996, end=2023,
)
meta = fetch_country_metadata()
real = set(meta.loc[~meta["is_aggregate"], "iso3"])
panel = panel[panel["iso3"].isin(real)].copy()
panel = panel.merge(meta[["iso3", "region", "income"]], on="iso3", how="left")

print(f"{len(panel):,} country-years × {panel['iso3'].nunique():,} countries")
panel.head()


## 2. Transformations

In [ ]:
panel["log_gdp_pc"]     = np.log(panel["NY.GDP.PCAP.KD"].replace(0, np.nan))
panel["log_pop"]        = np.log(panel["SP.POP.TOTL"].replace(0, np.nan))
panel["trade_share"]    = panel["NE.TRD.GNFS.ZS"]
panel["invest_share"]   = panel["NE.GDI.TOTL.ZS"]
# Rename WGI regressors for readability in the coef table.
panel = panel.rename(columns={
    "RL.EST": "rule_of_law",
    "CC.EST": "corruption_control",
    "GE.EST": "gov_effectiveness",
})


## 3. Descriptive: within-country movement in institutions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, col in zip(axes, ["rule_of_law", "corruption_control", "gov_effectiveness"]):
    sample = panel.dropna(subset=[col])
    sample.groupby("year")[col].mean().plot(ax=ax, marker="o")
    ax.set_title(col)
    ax.set_xlabel("")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Cross-country mean (WGI z-score)")
fig.suptitle("Global mean governance scores, 1996-2023")
fig.tight_layout()


## 4. Two-way FE regression

In [ ]:
res = fixed_effects_panel(
    panel,
    dependent="log_gdp_pc",
    regressors=["rule_of_law", "corruption_control", "gov_effectiveness",
                "trade_share", "invest_share", "log_pop"],
    entity_effects=True, time_effects=True,
    name="RQ1  -  Governance → log(GDP per capita), 1996-2023",
)
print(res.as_markdown())


## 5. Heterogeneity  -  is the effect stronger for lower-income countries?

In [ ]:
results = {}
for grp in ["Low income", "Lower middle income", "Upper middle income", "High income: OECD", "High income: nonOECD"]:
    sub = panel[panel["income"] == grp]
    if sub["iso3"].nunique() < 8:
        continue
    try:
        r = fixed_effects_panel(
            sub, dependent="log_gdp_pc",
            regressors=["rule_of_law", "trade_share", "invest_share", "log_pop"],
            entity_effects=True, time_effects=True,
            name=f"By income group: {grp}",
        )
        results[grp] = r.coef_table.loc["rule_of_law"]
    except Exception as e:
        print(f"  skip {grp}: {e}")

pd.DataFrame(results).T[["coef", "stderr", "pvalue"]].round(3)


## 6. Save

In [ ]:
from pathlib import Path
Path("../results").mkdir(exist_ok=True)
res.coef_table.to_csv("../results/rq1_institutions_coefs.csv")
